# دفتر محاكاة ساحة المشرع

المختبرات 1.5 إلى 1.7 · هيئة الرفادة للمدينة المضيفة (حالة افتراضية).

دفتر جاهز، لا تكتب فيه كودًا. تختار الخطة والظرف والبذرة في الخلية الثانية، ثم تشغّل كل الخلايا.

- **في Google Colab:** File ثم Upload notebook، واختر هذا الملف. ثم Runtime ثم Run all. أول تشغيل يثبّت JuPedSim في دقيقة أو اثنتين.
- **على حاسوبك:** `uv run plaza_sim.py --plan A --scenario base --seed 1` من مجلد `labs/sim`.
- **إن لم يعمل أي منهما:** النتائج الجاهزة في `labs/files/sim/runs.csv` هي نفسها.

الساحة 50 مترًا عرضًا و30 مترًا عمقًا. يصل الزوار من الطرف الجنوبي في موجات (قطار كل 150 ثانية) ويمشون إلى أقرب بوابة مفتوحة.
تُقاس الكثافة في مستطيل 8 × 5 أمتار أمام كل بوابة كل ثانية، ويؤخذ متوسط عشر ثوانٍ، كما تعرّفها خطة السلامة. الحد 2.0 شخص في المتر المربع.

In [ ]:
# Installs the simulation library when it is missing (Colab), then nothing more
import importlib.util, subprocess, sys
missing = [p for p, m in (("jupedsim==1.4.2", "jupedsim"), ("shapely", "shapely"), ("matplotlib", "matplotlib")) if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("ready")

In [ ]:
# Choose here, then run every cell
PLAN = "A"          # @param ["A", "B"]
SCENARIO = "base"   # @param ["base", "plus25", "gate_closed", "both"]
SEED = 1            # @param [1, 2, 3]

## النموذج

لا تغيّر هذه الخلية. فيها الساحة والخطتان والظروف الأربعة: `base` يوم معتاد، و`plus25` وافدون أكثر بالربع، و`gate_closed` بوابة مغلقة، و`both` الاثنان معًا.

In [ ]:
import random

import jupedsim as jps
import shapely

PLAZA_W, PLAZA_D, DT = 50.0, 30.0, 0.05
LIMIT = 2.0  # persons per square metre, from the safety plan (a decision of the authority)
PLANS = {
    "A": [(21.0, 3.0), (29.0, 3.0)],                # two central gates, 3 m each
    "B": [(10.0, 2.0), (25.0, 2.0), (40.0, 2.0)],   # three spread gates, 2 m each
}
CLOSED = {"A": 29.0, "B": 25.0}  # the gate that is closed in the stress test
SCENARIOS = {
    "base": (450, False),           # 450 visitors per train wave, every gate open
    "plus25": (563, False),         # 25 % more visitors (the peak forecast error)
    "gate_closed": (450, True),     # one gate closed
    "both": (563, True),            # both at once
}
WAVES, GAP, WAVE_LEN = 4, 150.0, 45.0


def gates_for(plan, scenario):
    per_wave, closed = SCENARIOS[scenario]
    gates = [g for g in PLANS[plan] if not (closed and g[0] == CLOSED[plan])]
    return gates, per_wave


def run(plan, scenario="base", seed=1, area=(8.0, 5.0), window=10):
    """One simulation. Returns a dict with the density series and the peaks per gate."""
    gates, per_wave = gates_for(plan, scenario)
    plaza = shapely.box(0, 0, PLAZA_W, PLAZA_D)
    corridors = [shapely.box(x - w / 2, PLAZA_D - 0.01, x + w / 2, PLAZA_D + 4) for x, w in gates]
    sim = jps.Simulation(model=jps.CollisionFreeSpeedModel(),
                         geometry=shapely.union_all([plaza] + corridors), dt=DT)
    targets = []
    for x, w in gates:
        exit_id = sim.add_exit_stage(shapely.box(x - w / 2, PLAZA_D + 3, x + w / 2, PLAZA_D + 4))
        targets.append((sim.add_journey(jps.JourneyDescription([exit_id])), exit_id))
    aw, ad = area
    boxes = [(x - aw / 2, x + aw / 2, PLAZA_D - ad, PLAZA_D) for x, _ in gates]
    rng = random.Random(seed)
    arrivals = sorted(k * GAP + rng.random() * WAVE_LEN for k in range(WAVES) for _ in range(per_wave))
    xs = [rng.uniform(1, PLAZA_W - 1) for _ in arrivals]
    nxt, step, series = 0, 0, []
    while True:
        t = step * DT
        while nxt < len(arrivals) and arrivals[nxt] <= t:
            x = xs[nxt]
            g = min(range(len(gates)), key=lambda i: abs(gates[i][0] - x))
            try:
                sim.add_agent(jps.CollisionFreeSpeedModelAgentParameters(
                    journey_id=targets[g][0], stage_id=targets[g][1],
                    position=(x, rng.uniform(0.4, 2.0)), radius=0.2, desired_speed=1.2))
            except Exception:
                xs[nxt] = rng.uniform(1, PLAZA_W - 1)  # the spot is taken: try another next step
                break
            nxt += 1
        sim.iterate()
        step += 1
        if step % int(round(1 / DT)) == 0:
            pts = [a.position for a in sim.agents()]
            series.append([sum(1 for (px, py) in pts if x0 <= px <= x1 and y0 <= py <= y1) / (aw * ad)
                           for x0, x1, y0, y1 in boxes])
        if nxt >= len(arrivals) and sim.agent_count() == 0:
            break
    smooth = []
    for i in range(len(series)):
        part = series[max(0, i - window + 1):i + 1]
        smooth.append([sum(r[g] for r in part) / len(part) for g in range(len(gates))])
    peaks = [round(max(r[g] for r in smooth), 2) for g in range(len(gates))]
    above = [sum(1 for r in smooth if r[g] > LIMIT) for g in range(len(gates))]
    return {"plan": plan, "scenario": scenario, "seed": seed, "gates": [x for x, _ in gates],
            "visitors": per_wave * WAVES, "seconds": len(series), "smooth": smooth,
            "peaks": peaks, "above": above}

In [ ]:
result = run(PLAN, SCENARIO, SEED)
print(f"plan {PLAN}, scenario {SCENARIO}, seed {SEED}: {result['visitors']} visitors, plaza clear after {result['seconds']} s")
for i, (x, peak, above) in enumerate(zip(result["gates"], result["peaks"], result["above"]), 1):
    print(f"G{i} (x = {x:g} m): peak density {peak:.2f} persons/m2, {above} s above the limit of {LIMIT}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
for g in range(len(result["gates"])):
    ax.plot([row[g] for row in result["smooth"]], label=f"G{g + 1}")
ax.axhline(LIMIT, color="red", linestyle="--", label="limit 2.0")
ax.set_xlabel("second"); ax.set_ylabel("persons per m2 (10 s mean)")
ax.set_title(f"Plan {PLAN}, {SCENARIO}, seed {SEED}")
ax.legend(); plt.show()

## ما تسجله

في جدول المقارنة: قمة الكثافة لكل بوابة، والثواني فوق الحد، والبذرة. اقرأ المنحنى أيضًا: متى ترتفع الكثافة؟ بعد كل قطار؟ وهل تعود إلى الصفر قبل القطار التالي؟

مهمة إضافية (1.5): غيّر مساحة القياس في السطر `run(PLAN, SCENARIO, SEED, area=(4.0, 3.0))` وانظر كيف تتغير القمة للتشغيل نفسه.